In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.set_printoptions(suppress=True)

In [12]:
from simulators import NestedModelFamily, ContextManager
from simulators.benchmarks import DDM, RDM, CDM
from adapters import Adapter

# Metas

In [13]:
ddm_intrinsics = ["v", "a", "tau", "s_v", "s_tau", "decay"]

# Priors

In [14]:
ddm_priors = {
    "v":     {"intercept": lambda: np.random.gamma(3.0, 0.8),
              "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":     {"intercept": lambda: np.random.gamma(10.0, 0.3),
              "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":   {"intercept": lambda: np.random.gamma(3.0, 0.2),
              "slope":     lambda: 0.0},
    "s_v":   {"intercept": lambda: np.random.gamma(1.0, 0.2),
              "slope":     lambda: 0.0},
    "s_tau": {"intercept": lambda: np.random.uniform(0.0, 0.4),
              "slope":     lambda: 0.0},
    "decay": {"intercept": lambda: np.random.gamma(1.0, 0.4),
              "slope":     lambda: 0.0},
}

# Context Manager

In [15]:
context_manager = ContextManager()

# Model Family

In [22]:
model_family = NestedModelFamily(
    name="DDM",
    model=DDM(),
    context_manager=context_manager,
    prior_fun=ddm_priors,
    intrinsic_params=ddm_intrinsics,
)

In [27]:
samples = model_family.batch_sample(
    batch_size=10,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau", "s_v", "decay"},
        fixed_intrinsics={"s_tau"}
    ),
    min_num_obs=20,
    max_num_obs=500,
    flatten_param_outputs=False
)

(9, 6) (9, 6)
(21, 6) (21, 6)
(3, 6) (3, 6)
(6, 6) (6, 6)
(15, 6) (15, 6)
(24, 6) (24, 6)
(18, 6) (18, 6)
(9, 6) (9, 6)
(30, 6) (30, 6)
(15, 6) (15, 6)


In [28]:
samples["param_masks"].shape

(10, 30, 6)

In [29]:
samples["param_matrices"].shape

(10, 30, 6)

In [30]:
samples["regressor_masks"].shape

(10, 30)

In [31]:
samples

{'model_names': ['DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM'],
 'design_configs': [{'u_0': ['a', 'tau', 's_v', 'decay'],
   'u_1': ['tau'],
   'u_2': ['v', 'a', 'tau', 's_v', 'decay']},
  {'u_0': ['v', 'a', 'decay'],
   'u_1': ['v', 'a', 'tau', 's_v', 'decay'],
   'u_2': ['v', 'a', 's_v'],
   'u_3': ['s_v'],
   'u_4': ['v', 'tau', 's_v', 'decay'],
   'u_5': ['decay'],
   'u_6': ['v', 'tau']},
  {'u_0': ['v', 'a', 'tau', 's_v', 'decay']},
  {'u_0': ['v', 'a', 'decay'], 'u_1': ['v']},
  {'u_0': ['s_v'],
   'u_1': [],
   'u_2': ['a', 'tau', 's_v', 'decay'],
   'u_3': ['v', 'tau'],
   'u_4': ['a', 'tau', 's_v', 'decay']},
  {'u_0': ['v', 's_v', 'decay'],
   'u_1': ['v', 'tau', 's_v', 'decay'],
   'u_2': ['decay'],
   'u_3': ['v', 'tau', 's_v', 'decay'],
   'u_4': ['a'],
   'u_5': ['v', 'decay'],
   'u_6': ['v'],
   'u_7': ['a']},
  {'u_0': ['v', 'tau', 's_v'],
   'u_1': ['v'],
   'u_2': ['a', 'tau', 'decay'],
   'u_3': ['v', 'a', 'tau', 's_v'],
  

# Adapter

In [32]:
adapter = Adapter()

In [33]:
design_matrices = adapter.convert_dtype(samples["design_matrices"], dtype=np.float32)
param_masks = adapter.convert_dtype(samples["param_masks"], dtype=np.float32)
rts = adapter.convert_dtype(samples["sim_data"]["rts"], dtype=np.float32)
choices = adapter.convert_dtype(samples["sim_data"]["choices"], dtype=np.float32)

In [34]:
batch_size, num_obs, num_cols = design_matrices.shape
print(batch_size, num_obs, num_cols)

10 450 30


In [35]:
y_rts_col = adapter.atleast_2d(rts, orientation="col")                 # (N, 1)
y_ch_col  = adapter.atleast_2d(rts,  orientation="col")

In [36]:
sim_data = adapter.concatenate([y_rts_col, y_ch_col], axis=1, dtype=np.float32, pad=False)

# RDM

In [37]:
rdm_priors = {
    "v":      {"intercept": lambda: np.random.gamma(3.0, 0.8),
               "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":      {"intercept": lambda: np.random.gamma(10.0, 0.3),
               "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":    {"intercept": lambda: np.random.gamma(3.0, 0.2),
               "slope":     lambda: np.random.normal(0.0, 0.2)},
    "decay":  {"intercept": lambda: np.random.gamma(1.0, 0.4),
               "slope":     lambda: np.random.normal(0.0, 0.2)},
}

In [38]:
family = NestedModelFamily(
    name="RDM",
    model=RDM(),
    context_manager=context_manager,
    prior_fun=rdm_priors,
    intrinsic_params=ddm_intrinsics,
)

In [39]:
num_alternatives = np.random.randint(2, 4, size=1)  # number of alternatives

context = {
    "correct_idx": lambda n: np.random.randint(0, num_alternatives, size=n),
    "num_alternatives": num_alternatives,
}

In [41]:
samples = family.batch_sample(
    batch_size=3,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau"},
        fixed_intrinsics={"decay"}
    ),
    min_num_obs=20,
    max_num_obs=500,
    context=context,
    flatten_param_outputs=False,
)

(30, 6) (30, 6)
(0, 6) (0, 6)
(0, 6) (0, 6)


In [42]:
samples

{'model_names': ['RDM', 'RDM', 'RDM'],
 'design_configs': [{'u_0': ['a'],
   'u_1': ['tau'],
   'u_2': ['v', 'a'],
   'u_3': ['a', 'tau'],
   'u_4': ['v', 'tau'],
   'u_5': [],
   'u_6': ['v', 'tau'],
   'u_7': ['v', 'tau'],
   'u_8': ['v', 'a'],
   'u_9': ['v', 'a']},
  {},
  {}],
 'design_matrices': array([[[0.        , 0.        , 0.        , ..., 0.92275022,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.35192188,
          0.        , 0.        ],
         [1.        , 0.        , 0.        , ..., 0.38057817,
          0.        , 0.        ],
         ...,
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ]],
 
        [[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.      

# CDM

In [43]:
cdm_intrinsics = ["v_x", "v_y", "a", "tau", "decay"]

In [44]:
cdm_priors = {
    "v_x":   {"intercept": lambda: np.random.normal(0.0, 1.0),
              "slope":     lambda: np.random.normal(0.0, 0.5)},
    "v_y":   {"intercept": lambda: np.random.normal(0.0, 1.0),
              "slope":     lambda: np.random.normal(0.0, 0.5)},
    "a":     {"intercept": lambda: np.random.gamma(10.0, 0.3),
              "slope":     lambda: np.random.normal(0.0, 0.5)},
    "tau":   {"intercept": lambda: np.random.gamma(3.0, 0.2),
              "slope":     lambda: np.random.normal(0.0, 0.1)},
    "decay": {"intercept": lambda: np.random.gamma(1.0, 0.4),
              "slope":     lambda: np.random.normal(0.0, 0.1)},
}

In [45]:
family = NestedModelFamily(
    name="CDM",
    model=CDM(),
    context_manager=context_manager,
    prior_fun=cdm_priors,
    intrinsic_params=cdm_intrinsics,
)

In [46]:
out = family.sample(
    design_config=None,
    num_obs=200,
    num_regressors=3,
    max_num_regressors=6,
    max_num_categories=4,   # dummy blocks up to 3 cols per regressor
    keep_intercept=True,
    discrete_prob=0.5,      # mix of continuous and dummy regressors
)

In [47]:
print("Keys:", out.keys())
print("design_matrix:", out["design_matrix"].shape)
print("param_mask:", out["param_mask"].shape)
print("param_matrix:", out["param_matrix"].shape)
print("sim rts/choices:", out["sim_trials"]["rts"].shape, out["sim_trials"]["choices"].shape)

Keys: dict_keys(['model_name', 'design_config', 'design_matrix', 'param_mask', 'param_matrix', 'sim_trials', 'discrete_mask', 'regressor_mask', 'max_num_regressors', 'keep_intercept'])
design_matrix: (200, 10)
param_mask: (50,)
param_matrix: (50,)
sim rts/choices: (200,) (200,)


In [49]:
batch = family.batch_sample(
    batch_size=4,
    num_obs=None,                 # randomized per item
    num_regressors=None,          # randomized per item
    min_num_obs=100,
    max_num_obs=250,
    min_num_regressors=0,
    max_num_regressors=5,
    max_num_categories=4,
    keep_intercept=True,
    discrete_prob=0.6,
    flatten_param_outputs=False,
)

(4, 5) (4, 5)
(10, 5) (10, 5)
(7, 5) (7, 5)
(16, 5) (16, 5)


In [44]:
print("Batched design_matrices:", batch["design_matrices"].shape)
print("Batched sim rts:", batch["sim_data"]["rts"].shape)
print("Batched sim choices:", batch["sim_data"]["choices"].shape)
print("num_obs per item:", batch["num_obs"].reshape(-1))
print("num_regressors per item:", batch["num_regressors"].reshape(-1))

Batched design_matrices: (4, 189, 16)
Batched sim rts: (4, 189)
Batched sim choices: (4, 189)
num_obs per item: [153. 172. 189. 175.]
num_regressors per item: [5. 5. 5. 5.]
